# Man-in-the-Middle — Blackout

The attacker interposed on the GCS<->vehicle link **blinds the GCS**:

- **Downlink:** all telemetry to the GCS is dropped (position, mission state,
  attitude). The attacker keeps `HEARTBEAT` alive so the link still *looks*
  healthy, and lets the `LOGIC_DONE` completion signal through so the sim can
  terminate.
- **Uplink:** every command from the GCS is dropped — the operator cannot
  reach the vehicle.

The GCS still tries its usual intervention (redirect east), but because it
never sees `MISSION_CURRENT`, the trigger never fires — and even if it did, the
command would be suppressed. The drone therefore flies its **full north
mission unbothered** while the GCS sees nothing. Gazebo still shows the true
flight (the Oracle/visualizer path is independent of the GCS link).

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import Intervention, MissionTrigger, SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.helpers.processes import SimProcess
from simulator.planner import AutoPlan, InterventionPlan
from simulator.runtime.mitm import BlackoutStrategy
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin and waypoints

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, -20, 0, 0)
cruise_alt = 10.0  # m
model = Model.IRIS
sysid = 1

mission_wps = ENU.list([(0, 0, 0), (0, 0, cruise_alt), (0, 40, cruise_alt)])

## Vehicle + MITM blackout

The GCS attempts to redirect the vehicle as in `5-GCS_intervention_mission.ipynb` the blackout MITM makes sure it never reaches the vehicle, 
and that the vehicle's telemetry never reaches the GCS either, so the trigger cannot even fire.

In [ ]:
mission_path = DATA_PATH / "missions" / "gcs_intervention.waypoints"

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=str(mission_path),
    firmware=model.firmware,
)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
    # mitm=BlackoutStrategy(),
)


# GCS

In [ ]:
gcs = SimGCS(name=f"{Color.BLUE.name}_{Color.BLUE.emoji}")
target = ENU(x=10, y=10, z=cruise_alt)
gcs.add_vehicle(vehicle)
gcs.intervene(
    vehicle,
    Intervention(
        # Take over once the drone is on its final mission item (the NAV_LAND)
        # AND has actually descended below 7 m - i.e. the landing is underway.
        # Ignores the takeoff climb through 7 m, and is unaffected by speedup.
        trigger=MissionTrigger(descending_below=cruise_alt - 3.0),
        plan=InterventionPlan.from_relative_path(
            relative_path=[home.to_rel(target).unpose()],
            enu_origin=enu_origin,
            relative_home=home,
            firmware=model.firmware,
            land=False,
        ),
    ),
)

## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
target_marker = GazMarker(
    name="target",
    group="targets",
    pos=target,
    color=Color.GREEN,
)
gaz.markers.append(target_marker)
gaz.markers.append(origin_marker)
gaz.markers


## Oracle + Simulator

In [ ]:
orac = Oracle()

orac.add_gcs(gcs)


simulator = Simulator(
    oracle=orac,
    visualizer=gaz,
    verbose=2,
    terminals=[SimProcess.MITM, SimProcess.LOGIC, SimProcess.GCS],
    speedup=3,
)

simulator.preview()

In [ ]:
simulator.run()


## What to observe

- The drone flies the **full north mission** and does **not** turn east.
- The MITM's own terminal streams `MITM downlink: dropped GLOBAL_POSITION_INT`
  (and `MISSION_CURRENT`, `ATTITUDE`, ...) continuously — direct, live proof
  the blackout is suppressing telemetry as it happens, not just after the fact
  in a log. There's no uplink traffic to drop: the GCS never saw the trigger
  seq, so it never attempted the redirect command at all.
- `simulator/logs/mitm/mitm_1.log` — `strategy=BlackoutStrategy`.
- `simulator/logs/GCSs/GCS_BLUE_*.log` — **no** `GCS intervention` line (the
  GCS never saw `MISSION_CURRENT` reach its trigger).
- The run still terminates cleanly because `HEARTBEAT` and `LOGIC_DONE` are
  allowed through.